# Sesión 2.2 — Ampliaciones y ejercicios


## Preparación de datos y librerías

Cargamos de nuevo las librerías y la tabla de teléfonos para que esta parte pueda ejecutarse por separado.


In [ ]:
from pathlib import Path  # Construye rutas portables a las carpetas y archivos.
import numpy as np  # Aporta funciones numéricas y herramientas para simulación.
import pandas as pd  # Permite leer y manipular tablas de datos.
import matplotlib.pyplot as plt  # Permite crear y mostrar gráficos.
from scipy import stats  # Contiene distribuciones y pruebas estadísticas.
from IPython.display import display  # Muestra tablas con formato dentro del notebook.
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW  # Facilita intervalos para diferencias de medias independientes.
from graficos_apareados import graficos_apareados  # Importa los gráficos de pares desde el módulo reutilizable.

DATA_PATH = Path.cwd().parent.parent / "datos"  # Localiza la carpeta de datos desde la ruta de trabajo.

datos = pd.read_csv(DATA_PATH / 'Mobiles.txt', sep=';', na_values=['NA', ''])  # Lee la tabla de móviles y convierte esos marcadores en ausentes.
display(datos)  # Muestra los datos para inspeccionar sus columnas y filas.

## Comparación de dos medias

### Muestras independientes

Comparamos el importe medio de la factura entre quienes tienen contrato `Plan` y `Prepaid`. Preparamos los dos grupos eliminando las facturas ausentes; después visualizamos sus distribuciones con un diagrama de cajas y un gráfico de puntos. Finalmente, estimamos la diferencia de medias y realizamos contrastes bilateral y unilateral por separado. **El orden importa:** en todos los cálculos la diferencia es la media de `Plan` menos la de `Prepaid`.

Recurso para explorar distribuciones muestrales: http://onlinestatbook.com/stat_sim/sampling_dist/

In [ ]:
grupos = [(str(nombre), parte['Bill'].dropna()) for nombre, parte in datos.groupby('PlanType', sort=True)]  # Agrupa las facturas por contrato y elimina ausentes de cada grupo.
(nombre1, grupo1), (nombre2, grupo2) = grupos  # Separa las etiquetas y los datos de los dos grupos.
print('Orden de la diferencia de medias:', nombre1, '-', nombre2)  # Deja claro el orden usado para restar las medias.
print(f'Tamaños de los grupos: {nombre1} = {len(grupo1)}, {nombre2} = {len(grupo2)}')  # Informa cuántos datos válidos hay en cada grupo.

#### Diagrama de cajas por tipo de contrato

El **boxplot** resume la mediana (línea central), el rango intercuartílico (caja), la extensión habitual de los datos (bigotes) y los valores atípicos (puntos). Permite comparar la posición y la dispersión de las facturas en ambos grupos, pero no muestra todas las observaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico.
ax.boxplot([grupo1, grupo2], tick_labels=[nombre1, nombre2], patch_artist=True,  # Dibuja una caja por grupo con las etiquetas correspondientes.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Establece color y borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Define el estilo de bigotes y extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Destaca la mediana de cada grupo.
ax.set_title('Facturas por tipo de contrato: diagrama de cajas', fontsize=20)  # Añade un título descriptivo.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta el eje con la variable medida.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el diagrama de cajas.

#### Gráfico de puntos (stripchart)

El **stripchart** representa cada factura individual. El pequeño desplazamiento horizontal aleatorio evita que los puntos con importes iguales queden totalmente superpuestos; **no cambia** el valor de la factura ni el grupo de cada estudiante.

In [ ]:
rng_puntos = np.random.default_rng(12345)  # Fija una semilla para que el desplazamiento de puntos sea reproducible.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y el eje para el gráfico.
for posicion, grupo in enumerate((grupo1, grupo2), start=1):  # Recorre los dos grupos y asigna a cada uno una posición horizontal.
    x = posicion + rng_puntos.uniform(-0.12, 0.12, size=len(grupo))  # Añade una pequeña dispersión horizontal para evitar solapamientos.
    ax.scatter(x, grupo, s=65, color='#9fc3e4', edgecolor='black',  # Dibuja una marca por factura con color y borde.
               linewidth=0.8, alpha=0.7)  # Ajusta el grosor del borde y la transparencia de los puntos.
ax.set_xlim(0.5, 2.5)  # Deja espacio a ambos lados de las dos columnas de puntos.
ax.set_xticks([1, 2], [nombre1, nombre2])  # Etiqueta cada posición con el nombre del grupo.
ax.set_title('Facturas por tipo de contrato: gráfico de puntos', fontsize=20)  # Añade un título explicativo.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta el eje vertical con la variable.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas.
plt.show()  # Muestra el gráfico de puntos.

#### IC del 95 % para la diferencia de medias

Estimamos la diferencia poblacional en el orden **Plan − Prepaid**. Con muestras independientes y suponiendo varianzas poblacionales iguales, el intervalo bilateral se calcula como

$$
(\bar{x}_1-\bar{x}_2)\;\pm\;t_{1-\alpha/2,\,n_1+n_2-2}\;s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}},
\qquad
s_p^2=\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}.
$$

Aquí $s_p$ es la desviación típica combinada. El intervalo estima $\mu_{Plan}-\mu_{Prepaid}$: si es completamente positivo, los datos son compatibles con una media mayor en Plan; si incluye 0, no se distingue una diferencia al nivel correspondiente. Requiere independencia y varianzas iguales; la asimetría y los valores atípicos aconsejan cautela.

In [ ]:
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW  # Importa herramientas para comparar medias y calcular sus intervalos.

comparacion = CompareMeans(DescrStatsW(grupo1), DescrStatsW(grupo2))  # Prepara la comparación usando los dos grupos en el orden establecido.
ic_95 = comparacion.tconfint_diff(alpha=0.05, usevar='pooled')  # Calcula el IC del 95 % suponiendo varianzas poblacionales iguales.
print(f'IC 95 % para la media de {nombre1} − la de {nombre2}: ({ic_95[0]:.2f}, {ic_95[1]:.2f})')  # Muestra los límites de la diferencia de medias.

**Cómo interpretarlo:** informa del intervalo en el orden **Plan − Prepaid** y en las unidades de factura. Si ambos límites son positivos, la media de Plan se estima mayor; si ambos son negativos, la media de Prepaid se estima mayor. Si el intervalo contiene 0, no hay evidencia suficiente de diferencia al 5 %. El intervalo trata sobre medias poblacionales, no sobre facturas individuales.

#### IC del 99 % para la diferencia de medias

Repetimos la estimación con confianza del 99 % y los mismos grupos. En la fórmula anterior se sustituye $\alpha=0{,}05$ por $\alpha=0{,}01$, usando el cuantil $t_{0{,}995,\,n_1+n_2-2}$. Al pedir mayor cobertura, el valor crítico y normalmente la amplitud aumentan. Compara sus límites con los del IC del 95 %; el orden sigue siendo Plan − Prepaid.

In [ ]:
ic_99 = comparacion.tconfint_diff(alpha=0.01, usevar='pooled')  # Calcula el IC del 99 % con el mismo método y los mismos grupos.
print(f'IC 99 % para la media de {nombre1} − la de {nombre2}: ({ic_99[0]:.2f}, {ic_99[1]:.2f})')  # Imprime los límites para compararlos con el IC del 95 %.

**Lectura del IC del 99 %:** aplica la misma regla de signos que para el 95 %, pero recuerda que se exige más confianza y el intervalo suele ser más ancho. Si el 99 % incluye 0 pero el 95 % no, la evidencia de diferencia es sensible al nivel de confianza escogido; no es una contradicción.

#### Contraste t bilateral: ¿son distintas las medias?

Contrastamos si las medias poblacionales de las facturas difieren:

$$
\left\{
\begin{aligned}
H_0 &: \mu_{Plan}-\mu_{Prepaid}=0 \\
H_1 &: \mu_{Plan}-\mu_{Prepaid}\ne0.
\end{aligned}
\right.
$$

Con varianzas iguales, el estadístico es

$$
t_{obs}=\frac{\bar{x}_{Plan}-\bar{x}_{Prepaid}}{s_p\sqrt{1/n_{Plan}+1/n_{Prepaid}}},
\qquad gl=n_{Plan}+n_{Prepaid}-2.
$$

Con $\alpha=0{,}05$, rechazamos $H_0$ si el valor p bilateral es menor que 0,05. Un estadístico positivo refleja una media muestral mayor en Plan. La prueba presupone independencia, varianzas iguales y, especialmente en muestras pequeñas, normalidad aproximada.

In [ ]:
resultado_bilateral = stats.ttest_ind(grupo1, grupo2, equal_var=True, alternative='two-sided')  # Contrasta bilateralmente las medias suponiendo varianzas iguales.
print(f'Prueba t bilateral ({nombre1} frente a {nombre2}):')  # Identifica el contraste y el orden de los grupos.
print(f't = {resultado_bilateral.statistic:.4f}; p-valor = {resultado_bilateral.pvalue:.4g}')  # Muestra el estadístico t y su valor p.
if resultado_bilateral.pvalue < 0.05:  # Compara el valor p con el nivel de significación del 5 %.
    print('Se rechaza la igualdad de medias al nivel del 5 %.')  # Informa de evidencia de medias distintas.
else:  # Se ejecuta cuando el valor p es al menos 0,05.
    print('No se rechaza la igualdad de medias al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar la igualdad.

**Interpretación:** interpreta el signo de $t$ respetando el orden Plan − Prepaid. Un valor p inferior a 0,05 indica evidencia de medias diferentes bajo los supuestos; no mide por sí solo el tamaño ni la importancia práctica de la diferencia. Si el intervalo del 95 % excluye 0, normalmente la decisión bilateral al 5 % coincide.

#### Contraste t unilateral: ¿es mayor la media de Plan?

La alternativa unilateral debe fijarse antes de examinar los resultados. Con el orden de grupos Plan − Prepaid:

$$
\left\{
\begin{aligned}
H_0 &: \mu_{Plan}-\mu_{Prepaid}\le0 \\
H_1 &: \mu_{Plan}-\mu_{Prepaid}>0.
\end{aligned}
\right.
$$

Se usa el mismo estadístico t y los mismos supuestos del contraste bilateral. Rechazamos $H_0$ si el valor p unilateral es menor que $\alpha$ (por ejemplo, 0,05). No se debe elegir la dirección después de ver qué grupo tuvo la media más alta.

In [ ]:
resultado_unilateral = stats.ttest_ind(grupo1, grupo2, equal_var=True, alternative='greater')  # Contrasta si la media del primer grupo supera la del segundo.
print(f'Prueba t unilateral (media de {nombre1} mayor que la de {nombre2}):')  # Describe la dirección de la alternativa.
print(f't = {resultado_unilateral.statistic:.4f}; p-valor = {resultado_unilateral.pvalue:.4g}')  # Muestra estadístico y valor p unilateral.
if resultado_unilateral.pvalue < 0.05:  # Comprueba significación al 5 % en la dirección planteada.
    print(f'Se rechaza H₀: hay evidencia de que la media de {nombre1} es mayor.')  # Expresa la decisión en el contexto de los grupos.
else:  # Se ejecuta si el valor p no alcanza el umbral.
    print(f'No hay evidencia suficiente de que la media de {nombre1} sea mayor.')  # Evita concluir que la media es necesariamente menor.

**Interpretación:** una prueba unilateral solo responde a la dirección definida de antemano. Si $p<0{,}05$, hay evidencia a favor de que Plan tiene mayor media; si no, no se acredita esa dirección. No conviertas un resultado no significativo en evidencia de que Prepaid sea mayor.

#### Microejercicio: comparar medias sin suponer varianzas iguales

En el ejemplo se usa Student con varianzas iguales. Repite el intervalo bilateral del 99 % con Welch, que no exige ese supuesto. Completa los argumentos de `tconfint_diff`; en esta función `usevar='unequal'` selecciona Welch. Compara el resultado con `ic_99` y comenta si ambos intervalos llevan a la misma conclusión respecto a una diferencia nula.

In [ ]:
try:
    # Microejercicio: reutiliza la comparación anterior y cambia el supuesto de varianzas.
    ic_welch_99 = comparacion.tconfint_diff(alpha=..., usevar=...)  # Completa alfa para un IC del 99 % y elige Welch.
    print(f'IC Welch del 99 %: ({ic_welch_99[0]:.2f}, {ic_welch_99[1]:.2f})')  # Muestra el intervalo que no presupone varianzas iguales.
    print(f'IC Student del 99 %: ({ic_99[0]:.2f}, {ic_99[1]:.2f})')  # Compara con el IC calculado bajo varianzas iguales.
    # Comprueba qué intervalo contiene 0 y si las conclusiones coinciden.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

### Datos apareados

En la encuesta de estudiantes, cada fila corresponde a la misma persona en **ambas** mediciones:

- `Sex`: sexo de la persona.
- `HrsSleep`: horas de sueño cada noche.
- `SQpick`: letra elegida entre Q y S.
- `Height`: altura en pulgadas.
- `RandNumb`: número entero entre 1 y 10 elegido al azar.
- `Fastest`: máxima velocidad a la que ha conducido, en millas por hora.
- `RtSpan`: longitud de la palma derecha, en centímetros.
- `LftSpan`: longitud de la palma izquierda, en centímetros.
- `Form`: orden en que se presentaron las opciones Q y S.

#### Lectura de datos y selección de pares completos

`pd.read_csv(..., sep='\t')` lee el fichero separado por tabulaciones; `display` muestra las filas para inspeccionarlas. Seleccionamos las columnas `LftSpan` y `RtSpan` y aplicamos `dropna()` **a ambas a la vez**: si a una persona le falta una medida, excluimos sus dos valores para no romper el emparejamiento. Mostramos cuántos pares completos quedan. Después veremos cada gráfico y cada prueba en una celda distinta.

Descripción de la encuesta: http://sites.stat.psu.edu/~dhunter/250h/data/readme.html

In [ ]:
PennState1 = pd.read_csv(DATA_PATH / 'PennState1.txt', sep='\t')  # Lee la encuesta separada por tabulaciones.
display(PennState1)  # Muestra la tabla para inspeccionar columnas y observaciones.

# Conservamos únicamente a quienes tienen medidas de ambas palmas.
pares = PennState1[['LftSpan', 'RtSpan']].dropna()  # Selecciona ambas medidas a la vez y elimina personas con algún dato ausente.
print(f'Personas con ambas medidas: {len(pares)}; pares incompletos excluidos: {len(PennState1) - len(pares)}')  # Informa los pares válidos y los excluidos.

#### Diagrama de cajas de las longitudes

Comparamos las **distribuciones** de las palmas izquierda y derecha: la línea de cada caja es la mediana, la caja abarca el 50 % central de los valores, los bigotes muestran la extensión habitual y los puntos aislados señalan posibles valores atípicos. Sirve para ver posición y dispersión de cada lado, pero **no permite saber qué dos medidas corresponden a la misma persona**; para eso usamos el gráfico siguiente.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura amplia y sus ejes.
ax.boxplot([pares['LftSpan'], pares['RtSpan']], tick_labels=['Izquierda', 'Derecha'],  # Dibuja una caja por cada lado de la palma.
           patch_artist=True,  # Permite colorear el interior de las cajas.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Establece el relleno y el borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Define los estilos de bigotes y extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Resalta la mediana de cada distribución.
ax.set_title('Longitud de las palmas: diagrama de cajas', fontsize=20)  # Añade el título.
ax.set_ylabel('Palma (cm)', fontsize=16)  # Etiqueta la medida y sus unidades.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
plt.show()  # Muestra el diagrama de cajas.

#### Gráfico de dispersión de las medidas emparejadas

Cada punto representa a **una persona**: su longitud izquierda es la coordenada horizontal y la derecha la vertical. La línea granate es la bisectriz $y=x$: un punto **encima** indica palma derecha mayor; uno **debajo**, palma izquierda mayor; sobre la línea, igualdad. Así vemos a la vez la relación entre ambas medidas y las diferencias dentro de cada par. Los puntos superpuestos pueden corresponder a varias personas; el gráfico no sustituye al contraste.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico de dispersión.
ax.scatter(pares['LftSpan'], pares['RtSpan'], s=65, color='#9fc3e4',  # Dibuja cada persona con izquierda en X y derecha en Y.
           edgecolor='black', linewidth=0.8)  # Añade bordes para diferenciar puntos.
limites = [pares.min().min(), pares.max().max()]  # Obtiene el mínimo y máximo comunes para la línea de referencia.
ax.plot(limites, limites, color='#800000', linewidth=1.8)  # Dibuja y=x, donde ambas medidas son iguales.
ax.set_title('Comparación de palmas por persona', fontsize=20)  # Añade el título.
ax.set_xlabel('Palma izquierda (cm)', fontsize=16)  # Etiqueta el eje X.
ax.set_ylabel('Palma derecha (cm)', fontsize=16)  # Etiqueta el eje Y.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
plt.show()  # Renderiza el gráfico.

#### Prueba t de muestras independientes: ejemplo incorrecto

Para comparar métodos, calculamos lo que daría `stats.ttest_ind` al tratar las mediciones de la mano derecha y de la izquierda como si procediesen de **personas distintas**. Plantearíamos un contraste bilateral de igualdad de medias poblacionales:

$$
\left\{
\begin{aligned}
H_0 &: \mu_{\mathrm{derecha}} = \mu_{\mathrm{izquierda}} \\
H_1 &: \mu_{\mathrm{derecha}} \ne \mu_{\mathrm{izquierda}}
\end{aligned}
\right.
$$

`equal_var=True` añade el supuesto de varianzas poblacionales iguales. **Aquí no se cumple el supuesto de independencia entre grupos:** cada longitud derecha tiene su correspondiente longitud izquierda en la misma persona. Ignorar esta relación calcula un error estándar y un valor p para un diseño distinto; aunque se muestren por motivos didácticos, **no deben usarse para concluir**. Tras los gráficos se presenta la prueba válida.

In [ ]:
# SOLO PARA COMPARAR MÉTODOS: esta prueba no es válida para estas mediciones apareadas.
resultado_independiente = stats.ttest_ind(pares['RtSpan'], pares['LftSpan'], equal_var=True)  # Calcula deliberadamente el resultado incorrecto como si fueran grupos independientes.
print(f'Tratamiento incorrecto como muestras independientes: t = {resultado_independiente.statistic:.4f}; '  # Inicia el mensaje con el estadístico calculado.
      f'p-valor bilateral = {resultado_independiente.pvalue:.4g}')  # Completa el mensaje con el p-valor bilateral.
print('No usar este valor p para sacar conclusiones: las dos medidas pertenecen a las mismas personas.')  # Recuerda por qué no se debe interpretar este contraste.

### Gráficos para datos apareados

La función `graficos_apareados`, importada desde el módulo `graficos_apareados.py`, prepara cuatro vistas de los mismos pares completos: (1) relación entre ambas medidas y línea de igualdad; (2) Bland–Altman, con diferencia frente al promedio, sesgo medio y límites descriptivos de acuerdo; (3) diferencias individuales respecto a cero; (4) perfiles que unen las dos medidas de cada individuo.

En Bland–Altman, para cada par se representa $D_i=Y_i-X_i$ frente a $A_i=(X_i+Y_i)/2$. Las líneas de acuerdo son $\bar{D}\pm1{,}96s_D$; **no son** un intervalo de confianza para $\mu_D$. En los perfiles, cada línea une las dos mediciones de una misma persona y la línea granate gruesa une las medias de los dos lados. Primero mostraremos todos los pares y después, solo como ayuda visual, los que difieren tras redondear.

#### Fase 1: todos los pares completos

Dibujamos las cuatro vistas sin filtrar por el tamaño ni el signo de la diferencia. Las líneas horizontales grises en el gráfico de perfiles indican igualdad exacta entre ambas palmas; las inclinadas permiten identificar qué lado es mayor. En el gráfico de diferencias, el eje horizontal indica la **posición de cada par** en los datos utilizados, no el identificador original de la persona.

In [ ]:
graficos_apareados(PennState1)  # Genera los cuatro gráficos usando todos los pares completos de palmas.
plt.show()

#### Fase 2: pares con longitudes redondeadas distintas

Volvemos a dibujar las cuatro vistas, pero excluimos los pares cuyas longitudes derecha e izquierda quedan **iguales al redondearlas al entero**. Antes descartamos los pares incompletos. Esto reduce la superposición de líneas horizontales y facilita ver las diferencias individuales.

**Importante:** redondear y filtrar es solo un recurso visual, no equivale a exigir una diferencia mínima fija ni justifica un nuevo contraste sobre los pares seleccionados. La prueba apareada que aparece **a continuación** utiliza todos los pares completos.

In [ ]:
pares_redondeados = PennState1.dropna(subset=['LftSpan', 'RtSpan'])  # Mantiene solo personas con ambas medidas conocidas.
distintos_redondeados = pares_redondeados['RtSpan'].round() != pares_redondeados['LftSpan'].round()  # Marca pares cuyas medidas redondeadas difieren.
PennState1_sub = pares_redondeados.loc[distintos_redondeados]  # Crea el subconjunto solo para mejorar la visualización.
print(f'Pares seleccionados: {len(PennState1_sub)} de {len(pares_redondeados)}')  # Informa cuántos pares se muestran frente al total completo.
graficos_apareados(PennState1_sub, 'Pares con longitudes redondeadas distintas')  # Dibuja los gráficos para el subconjunto seleccionado.
plt.show()

#### Prueba t para muestras apareadas: comparación correcta

Cada persona aporta un par. Definimos $D_i=\mathrm{RtSpan}_i-\mathrm{LftSpan}_i$ (derecha menos izquierda) y contrastamos la media poblacional de las diferencias:

$$
\left\{
\begin{aligned}
H_0 &: \mu_D=0 \\
H_1 &: \mu_D\ne0.
\end{aligned}
\right.
$$

La prueba t apareada equivale a una prueba t de una muestra aplicada a $D_1,\ldots,D_n$:

$$
t_{obs}=\frac{\bar{D}}{s_D/\sqrt{n}},\qquad gl=n-1.
$$

Con $\alpha=0{,}05$, rechazamos $H_0$ si el valor p bilateral es menor que 0,05. El error estándar se calcula sobre las diferencias dentro de cada persona, no tratando las manos como muestras independientes. Se supone que los pares de personas distintas son independientes y que las diferencias son aproximadamente normales, sobre todo con muestras pequeñas.

In [ ]:
# Restamos las medidas de cada persona en el mismo orden que en la prueba.
diferencias_palma = pares['RtSpan'] - pares['LftSpan']  # Calcula, para cada persona, derecha menos izquierda.
resultado_apareado = stats.ttest_rel(pares['RtSpan'], pares['LftSpan'])  # Ejecuta la prueba t apareada sobre las dos mediciones emparejadas.
print(f'Pares: {len(pares)}; diferencia media (derecha − izquierda): {diferencias_palma.mean():.2f} cm')  # Informa el número de personas y la diferencia media.
print(f't = {resultado_apareado.statistic:.4f}; p-valor bilateral = {resultado_apareado.pvalue:.4g}')  # Muestra el estadístico t y el p-valor bilateral.
if resultado_apareado.pvalue < 0.05:  # Compara el p-valor con el nivel de significación del 5 %.
    print('Se rechaza H₀ al nivel del 5 %: hay evidencia de una diferencia media entre las palmas.')  # Comunica la decisión cuando el resultado es significativo.
else:  # Se ejecuta cuando el p-valor es al menos 0,05.
    print('No se rechaza H₀ al nivel del 5 %: no hay evidencia suficiente de una diferencia media.')  # Comunica que no hay evidencia suficiente contra H₀.

**Interpretación:** el estadístico t y el valor p se calculan a partir de las diferencias por persona. Si el IC del 99 % incluye 0, no se detecta una diferencia media al nivel bilateral del 1 %; si queda enteramente por encima o por debajo de 0, la dirección se interpreta como derecha mayor o menor, respectivamente. Una diferencia media no implica que todos los pares difieran en el mismo sentido.

#### Microejercicio: intervalo para una diferencia apareada

Partimos de $D_i=\mathrm{RtSpan}_i-\mathrm{LftSpan}_i$, definida para cada persona. El IC t bilateral para la diferencia media es

$$
\bar{D}\pm t_{1-\alpha/2,\,n-1}\frac{s_D}{\sqrt{n}}.
$$

Calcula el intervalo del **99 %** con una prueba t de una muestra sobre `diferencias_palma`. Comprueba si incluye 0 e interpreta el signo: valores positivos indican una longitud derecha mayor en promedio.

In [ ]:
try:
    # Microejercicio: intervalo t del 99 % para la media de las diferencias apareadas.
    resultado_palma_99 = stats.ttest_1samp(diferencias_palma, popmean=...)  # Contrasta si la diferencia media derecha−izquierda es cero.
    ic_palma_99 = resultado_palma_99.confidence_interval(confidence_level=...)  # Calcula el IC bilateral del 99 %.
    print(f'IC del 99 % para la diferencia media (derecha − izquierda): ({ic_palma_99.low:.3f}, {ic_palma_99.high:.3f})')
    # Comprueba si el intervalo incluye 0 e interpreta el signo en centímetros.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

## Ejercicio 2.2 — Retrasos de vuelos

Completa los cuatro apartados en orden. Distingue la comparación de **grupos independientes** (JFK frente a otros aeropuertos) del análisis **apareado** (salida y llegada del mismo vuelo). Expresa las diferencias en minutos y conserva el orden indicado en cada apartado.

#### Lectura de los datos de vuelos

Cargamos aquí el fichero necesario para el ejercicio 2.2. La tabla `vuelos_df` se reutiliza en los ejercicios 2.3 y 2.4; se lee una vez, justo antes de comenzar estos ejercicios. El fichero está separado por tabulaciones y `@`, `NA` y las celdas vacías se interpretan como valores ausentes.

In [ ]:
vuelos_df = pd.read_csv(DATA_PATH / 'flights.txt', sep='\t', na_values=['@', 'NA', ''])  # Lee vuelos_df desde el archivo tabulado y reconoce marcadores de datos ausentes.
display(vuelos_df.head())  # Muestra las primeras filas como comprobación de la carga.

#### 2.2.1 — Crear y revisar la categoría de origen

Crea `origin2`: asigna `JFK` cuando `origin` sea JFK y `Other` en el resto. Comprueba los recuentos de ambos grupos; las ausencias en `origin` deben seguir como ausentes, no clasificarse como `Other`.

In [ ]:
# Pistas: usa np.where(vuelos_df['origin'].eq('JFK'), 'JFK', 'Other').
# Para que los orígenes ausentes no se conviertan en Other, conserva NA con loc o where.
# Guarda la clasificación en vuelos_df['origin2'] y revisa value_counts(dropna=False).

#### 2.2.2 — Comparar medias independientes: JFK − Other

Compara `arr_delay` entre JFK y Other. Elimina retrasos ausentes en cada grupo y usa Welch, que no exige igualdad de varianzas. Para contrastar $H_0:\mu_{JFK}-\mu_{Other}=0$ frente a una alternativa bilateral, el estadístico es

$$
t_{obs}=\frac{\bar{x}_{JFK}-\bar{x}_{Other}}{\sqrt{s^2_{JFK}/n_{JFK}+s^2_{Other}/n_{Other}}},
$$

con grados de libertad aproximados de Welch–Satterthwaite $\nu$. Calcula también el IC bilateral del 95 %, manteniendo el orden **JFK − Other**. Interpreta el signo y comprueba si el intervalo contiene 0. Al nivel $\alpha=0{,}05$, el contraste rechaza $H_0$ cuando su valor p bilateral es menor que 0,05.

In [ ]:
# Pistas:
# - Crea grupo_jfk y grupo_other con loc sobre origin2; aplica dropna() a arr_delay.
# - stats.ttest_ind(grupo_jfk, grupo_other, equal_var=False) ejecuta Welch.
# - Usa confidence_interval(confidence_level=0.95) en el resultado del contraste.
# Mantén el orden JFK − Other al mostrar e interpretar la diferencia.

#### 2.2.3 — Diferencia apareada e IC del 99,9 %

Cada vuelo tiene retraso de salida y llegada. Para cada vuelo completo calcula $D_i=arr\_delay_i-dep\_delay_i$ y el intervalo t para la diferencia media:

$$
\bar{D}\pm t_{1-\alpha/2,\,n-1}\frac{s_D}{\sqrt{n}},
\qquad \alpha=0{,}001\text{ para confianza del }99{,}9\%.
$$

Usa todos los pares completos, no dos muestras independientes. Expresa el intervalo en minutos y comprueba si incluye 0; un valor positivo representa, en promedio, un retraso de llegada mayor que el de salida.

In [ ]:
# Pistas: elimina filas sin arr_delay o dep_delay antes de restar.
# diferencias_vuelo = pares_vuelo['arr_delay'] - pares_vuelo['dep_delay'].
# Usa stats.ttest_1samp(diferencias_vuelo, popmean=0) y confidence_interval(confidence_level=0.999).
# Informa la diferencia como llegada menos salida y en minutos.

#### 2.2.4 — Gráfico de Bland–Altman

Selecciona los 10 000 primeros vuelos que cumplan `arr_delay < 180` y tengan ambos retrasos observados. Representa **llegada − salida** frente al promedio de ambos retrasos. Añade la diferencia media y los límites descriptivos $\bar{D}\pm1{,}96s_D$. Estos límites describen acuerdo/variabilidad, no un IC para la diferencia media.

In [ ]:
# Pistas:
# - Filtra vuelos_df por arr_delay < 180 y conserva arr_delay/dep_delay completos.
# - Para reutilizar la función importada, renombra esas columnas a LftSpan y RtSpan
#   (o indica columna_izquierda='dep_delay', columna_derecha='arr_delay').
# - Llama graficos_apareados(..., etiqueta_izquierda='Salida',
#   etiqueta_derecha='Llegada', unidades='min').
# Usa solo los 10 000 primeros vuelos que cumplan el filtro.

<div style="break-after: page; page-break-after: always;"></div>

## Comparación de dos varianzas (opcional)

Retomamos las facturas válidas de `Plan` y `Prepaid` preparadas anteriormente. Primero exploramos gráficamente la **dispersión**; después comparamos las **varianzas poblacionales** con una prueba F y calculamos, por separado, intervalos de confianza para el cociente de varianzas y el cociente de desviaciones típicas, siempre en el orden **Plan / Prepaid**.

La prueba F y sus intervalos requieren observaciones independientes en ambos grupos y poblaciones aproximadamente **normales**. La asimetría y los valores atípicos de las facturas aconsejan interpretar los resultados con cautela.

### Exploración gráfica de la dispersión

Dibujamos las facturas de `Plan` y `Prepaid` en **dos cajas sobre la misma escala**. Una caja más alta indica mayor amplitud entre los cuartiles; también conviene comparar los bigotes y los valores atípicos. Estas diferencias visuales son descriptivas y no bastan por sí mismas para afirmar que las varianzas poblacionales difieran.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura amplia y un eje para el boxplot.
ax.boxplot([grupo1, grupo2], tick_labels=[nombre1, nombre2], patch_artist=True,  # Dibuja cajas para los dos grupos con etiquetas.
           boxprops={'facecolor': '#9fc3e4', 'edgecolor': '#666666', 'linewidth': 1.2},  # Define relleno y borde de las cajas.
           whiskerprops={'color': '#666666'}, capprops={'color': '#666666'},  # Da estilo a los bigotes y a sus extremos.
           medianprops={'color': '#800000', 'linewidth': 1.8})  # Resalta las medianas.
ax.set_title('Facturas por tipo de contrato: comparación de la dispersión', fontsize=20)  # Añade título al gráfico.
ax.set_ylabel('Factura', fontsize=16)  # Etiqueta la variable de respuesta.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de los valores de los ejes.
plt.show()  # Muestra la comparación de dispersión.

### Función para comparar la variabilidad

Definimos `prueba_varianzas(a, b, confianza=0.95)` para reutilizarla en otros datos, como en el ejercicio siguiente. La función elimina los ausentes de cada grupo, calcula el estadístico y el valor p de un **contraste F bilateral**, y devuelve dos intervalos de confianza: uno para $\sigma_a^2/\sigma_b^2$ y otro para $\sigma_a/\sigma_b$. El **orden de los argumentos importa**: aquí `a` es `Plan` y `b` es `Prepaid`. A continuación examinamos sus resultados paso a paso.

In [ ]:
def prueba_varianzas(a, b, confianza=0.95):  # Define una función para comparar la dispersión de dos muestras independientes.
    """Contraste F bilateral e IC del cociente de varianzas y desviaciones."""  # Resume los resultados que devuelve la función.
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()  # Convierte ambos argumentos en series y elimina sus valores ausentes.
    gl1, gl2 = len(a)-1, len(b)-1  # Calcula los grados de libertad de numerador y denominador.
    cociente = a.var(ddof=1) / b.var(ddof=1)  # Calcula el cociente de varianzas muestrales en el orden a/b.
    pvalor = min(1, 2 * min(stats.f.cdf(cociente, gl1, gl2), stats.f.sf(cociente, gl1, gl2)))  # Calcula el p-valor bilateral usando la cola menor.
    alfa = 1-confianza  # Convierte el nivel de confianza en el nivel de significación.
    ic_var = (cociente/stats.f.ppf(1-alfa/2, gl1, gl2),  # Calcula el límite inferior del IC del cociente de varianzas.
              cociente/stats.f.ppf(alfa/2, gl1, gl2))  # Calcula el límite superior del intervalo.
    return {'F': cociente, 'p-valor': pvalor, 'IC varianzas': ic_var,  # Devuelve el cociente F, el p-valor y el IC de varianzas.
            'IC desviaciones': tuple(np.sqrt(ic_var))}  # Devuelve el IC del cociente de desviaciones típicas aplicando raíces.

#### Contraste F bilateral: ¿son iguales las varianzas?

Contrastamos las varianzas poblacionales en el orden Plan / Prepaid:

$$
\left\{
\begin{aligned}
H_0 &: \sigma^2_{Plan}=\sigma^2_{Prepaid} \\
H_1 &: \sigma^2_{Plan}\ne\sigma^2_{Prepaid}.
\end{aligned}
\right.
$$

El estadístico es

$$
F_{obs}=\frac{s^2_{Plan}}{s^2_{Prepaid}},\qquad gl_1=n_{Plan}-1,\quad gl_2=n_{Prepaid}-1.
$$

Un cociente cercano a 1 indica varianzas muestrales similares. Bajo $H_0$ y normalidad, $F_{obs}$ sigue una distribución F. Para $\alpha=0{,}05$, rechazamos $H_0$ si el valor p bilateral es menor que 0,05. La prueba es sensible a desviaciones de normalidad y a valores atípicos.

In [ ]:
resultado_varianzas = prueba_varianzas(grupo1, grupo2)  # Ejecuta el contraste F y calcula intervalos para Plan/Prepaid.
print(f'Contraste F bilateral ({nombre1} / {nombre2}): '  # Identifica el orden del cociente que se está informando.
      f'F = {resultado_varianzas["F"]:.4f}; p-valor = {resultado_varianzas["p-valor"]:.4g}')  # Muestra el estadístico y el p-valor.
if resultado_varianzas['p-valor'] < 0.05:  # Compara el p-valor con alfa = 0,05.
    print('Se rechaza la igualdad de varianzas al nivel del 5 %.')  # Informa de evidencia de varianzas distintas.
else:  # Se ejecuta si el p-valor es al menos 0,05.
    print('No se rechaza la igualdad de varianzas al nivel del 5 %.')  # Informa que no hay evidencia suficiente para rechazar igualdad.

**Interpretación:** compara $F$ con 1 y el valor p con $\alpha=0{,}05$. Un valor p bajo aporta evidencia contra la igualdad de varianzas; uno alto no demuestra que sean iguales. La asimetría de las facturas limita la confianza en este contraste F.

#### IC del 95 % para el cociente de varianzas

Sea $R=s^2_{Plan}/s^2_{Prepaid}$ y $F_{q;gl_1,gl_2}$ el cuantil $q$ de la distribución F. El intervalo bilateral de confianza $1-\alpha$ para el cociente poblacional es

$$
IC_{1-\alpha}\!\left(\frac{\sigma^2_{Plan}}{\sigma^2_{Prepaid}}\right)=\left[\frac{R}{F_{1-\alpha/2;gl_1,gl_2}},\;\frac{R}{F_{\alpha/2;gl_1,gl_2}}\right].
$$

Para el 95 %, $\alpha=0{,}05$. El valor 1 representa varianzas iguales; si el intervalo no lo contiene, la conclusión concuerda con rechazar la igualdad en un contraste bilateral al 5 %. Este procedimiento requiere grupos independientes y poblaciones aproximadamente normales.

In [ ]:
ic_var = resultado_varianzas['IC varianzas']  # Recupera el intervalo para el cociente de varianzas.
print(f'IC del 95 % para σ² de {nombre1} / σ² de {nombre2}: '  # Indica qué cociente y nivel de confianza se muestran.
      f'({ic_var[0]:.2f}, {ic_var[1]:.2f})')  # Completa e imprime los dos límites.

**Lectura:** comprueba si el cociente 1 cae dentro del intervalo. Si queda fuera, los datos indican una diferencia de dispersión bajo el modelo F; si queda dentro, no se descarta la igualdad. Interpreta este resultado con cautela por la no normalidad.

#### IC del 95 % para el cociente de desviaciones típicas

La desviación típica es la raíz cuadrada de la varianza. Por tanto, aplicamos la raíz a ambos límites del intervalo anterior:

$$
IC_{1-\alpha}\!\left(\frac{\sigma_{Plan}}{\sigma_{Prepaid}}\right)=\left[\sqrt{L},\sqrt{U}\right],
$$

siendo $L$ y $U$ los límites del IC para el cociente de varianzas. Un valor 1 representa dispersiones iguales; el orden sigue siendo Plan / Prepaid. Se mantienen los supuestos de normalidad e independencia.

In [ ]:
ic_desv = resultado_varianzas['IC desviaciones']  # Recupera el intervalo del cociente de desviaciones típicas.
print(f'IC del 95 % para σ de {nombre1} / σ de {nombre2}: '  # Identifica el parámetro y el orden del cociente.
      f'({ic_desv[0]:.2f}, {ic_desv[1]:.2f})')  # Muestra los límites con dos decimales.

**Lectura:** interpreta el cociente respecto a 1: por encima sugiere mayor desviación típica en Plan; por debajo, mayor en Prepaid. Si el intervalo contiene 1, no queda establecida una diferencia al nivel de confianza elegido.

#### Microejercicio: repetir la comparación de varianzas al 99 %

Usa `prueba_varianzas` con los mismos grupos `Plan` y `Prepaid`, pero eleva la confianza al 99 %. Completa el nivel y extrae los dos intervalos del resultado. ¿El intervalo de varianzas contiene 1? Compáralo con el contraste F al 1 %.

In [ ]:
try:
    # Microejercicio: repite el análisis de varianzas con confianza del 99 %.
    resultado_varianzas_99 = prueba_varianzas(grupo1, grupo2, confianza=...)  # Completa el nivel de confianza solicitado.
    ic_var_99 = resultado_varianzas_99['IC varianzas']  # Extrae el intervalo del cociente de varianzas.
    ic_desv_99 = resultado_varianzas_99['IC desviaciones']  # Extrae el intervalo del cociente de desviaciones típicas.
    print(f'F = {resultado_varianzas_99["F"]:.3f}; p = {resultado_varianzas_99["p-valor"]:.4g}')  # Muestra estadístico y p-valor.
    print(f'IC 99 % varianzas: ({ic_var_99[0]:.3f}, {ic_var_99[1]:.3f})')  # Imprime los límites de varianza.
    print(f'IC 99 % desviaciones: ({ic_desv_99[0]:.3f}, {ic_desv_99[1]:.3f})')  # Imprime los límites para desviaciones.
    # Interpreta si el cociente 1 es compatible con los datos.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

<div style="break-after: page; page-break-after: always;"></div>

## Ejercicio 2.3 — Comparar dispersión de retrasos por origen

Trabaja con los grupos `JFK` y `Other` definidos en el ejercicio anterior. En cada apartado, conserva ceros y valores negativos salvo que la instrucción pida filtrar solo los retrasos positivos.

### 2.3.1 — Boxplots de retraso por origen

Separa los retrasos de llegada conocidos en `JFK` y `Other` y dibuja ambos boxplots en la misma escala. Compara centro, dispersión, asimetría y valores atípicos. Este gráfico es descriptivo: no concluyas igualdad o diferencia de varianzas solo a partir de las cajas.

In [ ]:
# Pistas: crea las series JFK y Other con loc sobre origin2 y arr_delay.
# Aplica dropna() a los retrasos; conserva observaciones negativas y ceros.
# Dibuja ambas series con ax.boxplot([...], tick_labels=['JFK', 'Other']).
# Etiqueta el eje de retraso en minutos y comenta las diferencias descriptivas.

### 2.3.2 — Boxplots de retrasos positivos en escala logarítmica

Repite el gráfico tras filtrar `arr_delay > 0`. Usa escala vertical logarítmica para distinguir mejor la cola de retrasos largos. Explica por qué ceros y valores negativos no se pueden incluir en esa escala y recuerda que este filtro cambia la pregunta: describe solo los vuelos que llegaron tarde.

In [ ]:
# Pistas: filtra arr_delay > 0 y vuelve a separar JFK y Other.
# Dibuja los boxplots de los retrasos positivos y usa ax.set_yscale('log').
# Compara la dispersión visible con el gráfico anterior; la población analizada ahora es solo la de vuelos retrasados.

### 2.3.3 — Contraste F e IC del 99 % para JFK / Other

Contrasta la igualdad de varianzas de los retrasos originales (incluyendo ceros y negativos), en el orden JFK / Other:

$$
\left\{
\begin{aligned}
H_0 &: \sigma^2_{JFK}=\sigma^2_{Other} \\
H_1 &: \sigma^2_{JFK}\ne\sigma^2_{Other}.
\end{aligned}
\right.
$$

Compara las varianzas con `prueba_varianzas(..., confianza=0.99)`. Para $\alpha=0{,}01$, rechaza $H_0$ si el valor p bilateral es menor que 0,01; interpreta el IC del 99 % en el mismo orden y comprueba si incluye 1. Advierte que los retrasos son asimétricos y que la prueba F requiere normalidad.

In [ ]:
# Pistas: conserva todos los arr_delay observados de los grupos JFK y Other.
# Usa prueba_varianzas(grupo_jfk, grupo_other, confianza=0.99).
# Informa el orden del cociente, F, p-valor e intervalos; compáralos con 1 y alpha=0.01.
# Concluye con cautela: la prueba F es sensible a la falta de normalidad.

<div style="break-after: page; page-break-after: always;"></div>

## Comparación de dos proporciones

Estudiamos si la **proporción de estudiantes con contrato `Prepaid`** es la misma en los dos grupos de `Gender`. Primero construimos una tabla de frecuencias, después comparamos las proporciones con un gráfico de barras apiladas al 100 % y, por último, realizamos un contraste $\chi^2$. Todos los pasos utilizan únicamente estudiantes con `Gender` y `PlanType` conocidos.

### Tabla de frecuencias: género y tipo de contrato

La **tabla de contingencia** cuenta cuántos estudiantes tienen cada combinación de `Gender` (filas) y `PlanType` (columnas). `pd.crosstab` excluye las filas con valores ausentes en cualquiera de las dos variables. Para comparar la proporción de `Prepaid` entre géneros, fíjate en los casos de `Prepaid` **respecto al total de cada fila**, no solo en sus recuentos absolutos.

In [ ]:
t1 = pd.crosstab(datos['Gender'], datos['PlanType'])  # Cuenta cada combinación de género y tipo de contrato en una tabla de contingencia.
display(t1)  # Muestra los recuentos observados para comparar las proporciones.

### Gráfico de barras apiladas al 100 %: comparación de proporciones

Este gráfico representa las proporciones de `Plan` y `Prepaid` dentro de cada género. Cada barra tiene la misma altura (100 %), por lo que comparar visualmente la fracción de `Prepaid` resulta más directo que con los recuentos absolutos. Los segmentos muestran la composición de cada grupo, no el tamaño muestral; consulta la tabla de frecuencias para conocer cuántas personas hay en cada género. El gráfico es descriptivo y no sustituye al contraste χ².

In [ ]:
# Calcula las proporciones dentro de cada género para que las barras sumen 100 %.
proporciones_genero = t1.div(t1.sum(axis=1), axis=0)  # Divide cada recuento entre el total de su fila.

fig, ax = plt.subplots(figsize=(12, 7))  # Crea una figura y un eje para las barras apiladas.
proporciones_genero.plot(
    kind='bar', stacked=True, ax=ax,
    color=['#6f9fc8', '#9fc3e4'], edgecolor='#666666', linewidth=1.0
)  # Representa la composición de contratos dentro de cada género.
ax.set_title('Tipo de contrato por género: proporciones', fontsize=20)  # Añade un título descriptivo.
ax.set_xlabel('Género', fontsize=16)  # Etiqueta el eje horizontal.
ax.set_ylabel('Proporción dentro del género', fontsize=16)  # Indica que cada barra suma 1.
ax.set_ylim(0, 1)  # Fija la escala desde 0 hasta 100 %.
ax.set_yticks(np.linspace(0, 1, 6), [f'{valor:.0%}' for valor in np.linspace(0, 1, 6)])  # Muestra la escala como porcentajes.
ax.legend(title='Tipo de contrato', fontsize=12, title_fontsize=12)  # Identifica los segmentos apilados.
ax.tick_params(labelsize=14)  # Ajusta el tamaño de las etiquetas.
plt.tight_layout()  # Evita que los textos y la leyenda se solapen.
plt.show()  # Muestra el gráfico.

### Contraste Z de dos proporciones: ¿es igual la proporción de prepago?

Comparamos la proporción poblacional de `Prepaid` entre mujeres y hombres:

$$
\left\{
\begin{aligned}
H_0 &: \pi_1=\pi_2 \\
H_1 &: \pi_1\ne\pi_2
\end{aligned}
\right.
$$

Sean $x_1,x_2$ los recuentos de estudiantes con `Prepaid`, y $n_1,n_2$ los tamaños de los grupos. Las proporciones muestrales son $\hat p_1=x_1/n_1$ y $\hat p_2=x_2/n_2$. Bajo $H_0$, el error estándar usa la proporción agrupada:

$$
\hat p=\frac{x_1+x_2}{n_1+n_2},\qquad
\hat z=\frac{\hat p_1-\hat p_2}{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}\;\dot\sim\;N(0,1).
$$

En Python, `proportions_ztest([x1, x2], [n1, n2])` de `statsmodels.stats.proportion` calcula el estadístico $Z$ y el valor p bilateral aproximado (bajo la nula se agrupan las proporciones). Para $\alpha=0{,}05$, se rechaza $H_0$ si $|Z|>1{,}96$, o equivalentemente si $p<0{,}05$. Sin corrección de continuidad, en una tabla $2\times2$ se cumple $\chi^2=Z^2$. Se requiere independencia y tamaños muestrales no muy pequeños, con suficientes éxitos y fracasos esperados.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest  # Implementa el contraste Z aproximado para dos proporciones.

# Contraste bilateral: Prepaid en mujeres frente a hombres.
x1 = t1.loc['Female', 'Prepaid']  # Éxitos (Prepaid) del primer grupo.
n1 = t1.loc['Female'].sum()  # Tamaño del primer grupo.
x2 = t1.loc['Male', 'Prepaid']  # Éxitos (Prepaid) del segundo grupo.
n2 = t1.loc['Male'].sum()  # Tamaño del segundo grupo.
p1 = x1 / n1  # Proporción observada en mujeres.
p2 = x2 / n2  # Proporción observada en hombres.
z_obs, pvalor = proportions_ztest(count=[x1, x2], nobs=[n1, n2], value=0, alternative='two-sided', prop_var=False)  # Usa proporción agrupada bajo H₀.
print(f'Prepaid: mujeres = {p1:.3f} ({x1}/{n1}); hombres = {p2:.3f} ({x2}/{n2})')
print(f'Z = {z_obs:.4f}; p-valor bilateral = {pvalor:.4g}')
print(f'Equivalencia sin corrección: χ² = Z² = {z_obs**2:.4f} (gl = 1)')
if pvalor < 0.05:
    print('Se rechaza H₀ al nivel del 5 %: hay evidencia de una diferencia entre proporciones.')
else:
    print('No se rechaza H₀ al nivel del 5 %: no hay evidencia suficiente de diferencia.')

**Interpretación:** compara el valor p con 0,05 y verifica que las frecuencias esperadas sean razonables. Un resultado significativo aporta evidencia de asociación entre género y tipo de contrato, pero no explica su causa. Un resultado no significativo no prueba que las proporciones sean exactamente iguales.

#### Microejercicio: implementar el contraste Z con `proportions_ztest`

Usa los recuentos de `Prepaid` y los tamaños de muestra de mujeres y hombres. Aplica `proportions_ztest` con alternativa bilateral y proporción nula igual a cero para la diferencia. Compara el estadístico Z y el valor p con el resultado anterior; deberían coincidir. Interpreta la decisión al nivel del 5 %.

In [ ]:
try:
    from statsmodels.stats.proportion import proportions_ztest  # Importa el contraste Z de dos proporciones.

    # Microejercicio: contrasta la diferencia de proporciones con la aproximación normal.
    exitos = [t1.loc['Female', 'Prepaid'], t1.loc['Male', 'Prepaid']]  # Número de éxitos en cada grupo.
    tamanos = [t1.loc['Female'].sum(), t1.loc['Male'].sum()]  # Tamaños muestrales en el mismo orden.
    proporciones_genero = t1.div(t1.sum(axis=1), axis=0)  # Muestra las proporciones de cada categoría por género.
    display(proporciones_genero)
    z_micro, p_micro = proportions_ztest(exitos, tamanos, alternative='two-sided')  # Contraste Z bilateral con proporción agrupada bajo H₀.
    print(f'Z = {z_micro:.4f}; p-valor bilateral = {p_micro:.4g}')
    if p_micro < 0.05:
        print('Se rechaza H₀ al nivel del 5 %.')
    else:
        print('No se rechaza H₀ al nivel del 5 %.')
    # Compara con el resultado del contraste Z manual que aparece arriba.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Ejercicio 2.4 — Retrasos y fin de semana

Crea las variables derivadas cuidando los valores ausentes, visualiza la tabla conjunta y contrasta la igualdad de proporciones. Trabaja solo con observaciones conocidas en las dos variables de cada análisis.

### 2.4.1 — Crear fecha, fin de semana y categoría de retraso

Construye `date` desde año, mes y día con `pd.to_datetime`; crea `weekend` a partir del día de la semana. Define `arr_delay_cat` como retraso positivo (`arr_delay > 0`), conservando los valores ausentes como ausentes. Revisa algunas filas y los recuentos.

In [ ]:
# Pistas:
# - Combina year, month y day con pd.to_datetime(vuelos_df[['year', 'month', 'day']]).
# - weekend = vuelos_df['date'].dt.dayofweek >= 5.
# - Crea arr_delay_cat con arr_delay > 0 y conserva NA en los retrasos ausentes.
# Revisa display(vuelos_df[['date', 'weekend', 'arr_delay', 'arr_delay_cat']].head()).

### 2.4.2 — Barras apiladas al 100 %: retrasos y fin de semana

Calcula las proporciones de vuelos con y sin retraso dentro de cada grupo (`weekend` o día laborable) y represéntalas con barras apiladas al 100 %. Excluye filas con datos ausentes en cualquiera de las dos variables. Así se comparan directamente las proporciones, sin confundirlas con los distintos tamaños de grupo.

In [ ]:
# Pistas: crea tabla = pd.crosstab(vuelos_df['weekend'], vuelos_df['arr_delay_cat']).
# Normaliza por filas: proporciones = tabla.div(tabla.sum(axis=1), axis=0).
# Dibuja proporciones.plot(kind='bar', stacked=True) y fija el eje y entre 0 y 1.
# Etiqueta el eje vertical como porcentaje y añade la leyenda de categorías.

### 2.4.3 — Contraste Z de dos proporciones

Contrasta si la proporción de vuelos **sin retraso** es igual entre fines de semana (grupo 1) y días laborables (grupo 2):

$$
H_0:\pi_1=\pi_2,\qquad H_1:\pi_1\ne\pi_2.
$$

Cuenta como éxito cada vuelo con `arr_delay_cat == False`. Con $x_i$ éxitos de $n_i$ vuelos, el contraste bilateral usa la proporción agrupada bajo $H_0$:

$$
\hat p=\frac{x_1+x_2}{n_1+n_2},\qquad
Z=\frac{\hat p_1-\hat p_2}{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}\;\dot\sim\;N(0,1).
$$

En Python, `proportions_ztest([x1, x2], [n1, n2], alternative='two-sided')` devuelve $Z$ y el valor p aproximado. La aproximación requiere observaciones independientes y muestras suficientemente grandes.

In [ ]:
# Pistas: crea una tabla cruzada de weekend por arr_delay_cat, excluyendo ausentes.
# Cuenta como éxitos los vuelos sin retraso (arr_delay_cat == False) en cada grupo.
# Aplica proportions_ztest(count=[x_fin_semana, x_laborable],
#                           nobs=[n_fin_semana, n_laborable],
#                           alternative='two-sided').
# Informa Z y p-valor; compara p con 0.05 e interpreta en contexto.

<div style="break-after: page; page-break-after: always;"></div>

## Errores de tipo I y tipo II (opcional)

### Simulación del error de tipo I

Estimamos la proporción de resultados significativos **cuando las medias son iguales**. Generamos dos grupos de 100 retrasos normales con media 10 y desviación 3, repetimos la prueba 1 000 veces y calculamos la fracción de valores p menores de 0,05 (error de tipo I, α). Fijamos una semilla para repetir el experimento.

In [ ]:
n = 100  # Define el tamaño muestral de cada grupo simulado.
nsim = 1000  # Establece cuántas veces se repetirá el experimento.
p = []  # Inicializa una lista para guardar los p-valores de cada simulación.
rng = np.random.default_rng(12345)  # Fija una semilla para poder reproducir las simulaciones.
for i in range(nsim):  # Repite el experimento el número de veces indicado.
    RetrasosPrat = rng.normal(loc=10, scale=3, size=n)  # Genera el primer grupo normal con media 10 y desviación típica 3.
    RetrasosBarajas = rng.normal(loc=10, scale=3, size=n)  # Genera el segundo grupo bajo la misma media nula.
    tt = stats.ttest_ind(RetrasosPrat, RetrasosBarajas, equal_var=True)  # Aplica la prueba t independiente con varianzas iguales.
    p.append(tt.pvalue)  # Guarda el p-valor para esta repetición.
print('Error tipo I estimado (α):', np.mean(np.array(p) < 0.05))  # Estima la fracción de falsos rechazos cuando H₀ es cierta.

### Simulación del error de tipo II ($\beta$)

El error de tipo II es no rechazar $H_0$ cuando la alternativa es cierta. En la simulación se estima como la fracción de pruebas no significativas bajo una diferencia real:

$$
\hat{\beta}=\frac{\#\{p\text{-valores}\ge\alpha\}}{N_{sim}},\qquad \widehat{\text{potencia}}=1-\hat{\beta}.
$$

#### Ejercicio 2.5 — Estimar beta y potencia

Usa **393 observaciones por grupo**, diferencia de medias **0,2**, varianza **1** en ambos grupos y $\alpha=0{,}05$. Fija una semilla y repite la prueba t independiente muchas veces. Compara la potencia simulada con la calculada por `TTestIndPower` de `statsmodels`. Una pequeña discrepancia es esperable por variabilidad Monte Carlo.

In [ ]:
# Pistas:
# - Fija n=393, nsim (por ejemplo 1000) y rng = np.random.default_rng(12345).
# - En cada repetición genera grupos normales con medias 0 y 0.2, desviación 1.
# - Guarda el p-valor de stats.ttest_ind(..., equal_var=True).
# - Estima beta como la fracción de p-valores >= 0.05; potencia simulada = 1 - beta.
# - Calcula la potencia teórica con TTestIndPower().power(
#   effect_size=0.2, nobs1=393, alpha=0.05, ratio=1).
# Escribe y compara ambos resultados.

## Reto final integrador (opcional)

Compara los retrasos de llegada entre vuelos que salen de JFK y del resto de aeropuertos. Esta vez decide qué contraste de medias corresponde si no quieres suponer varianzas iguales; calcula también un IC del 95 % para la diferencia JFK − Other y usa `prueba_varianzas` como comprobación descriptiva adicional. Termina con una conclusión que integre tamaños muestrales, medias, intervalo, valor p y cautelas: los retrasos son asimétricos y la prueba F presupone normalidad.

**Pista:** reutiliza la clasificación `origin2`, los retrasos válidos y las funciones vistas en los apartados anteriores. Mantén el mismo orden de grupos en el contraste y en el intervalo.

In [ ]:
# Pistas: construye grupos arr_delay válidos de JFK y Other con loc y dropna().
# Resume tamaños y medias; usa stats.ttest_ind(..., equal_var=False) para Welch.
# Obtén IC bilateral del 95 % con confidence_interval() y conserva el orden JFK − Other.
# Reutiliza prueba_varianzas(...) como comparación complementaria, advirtiendo que los retrasos no son normales.
# Concluye en contexto: estimación, intervalo, p-valor, supuestos y relevancia práctica.